# 08 · Explorer: read anything

A playground for poking at the dataspace yourself. It gathers every **read** endpoint behind a few small helpers, with ready-made examples you can copy and change.
It does not depend on the other notebooks, but there is more to see after running them.

| Section | What you can read |
|---|---|
| 1 | Management API of the provider and consumer connectors: assets, policies, contract definitions, negotiations, agreements, transfers, EDRs, data planes, secrets, retirements |
| 2 | Catalogs: DSP catalog, a single dataset, federated catalog, filtered federated catalog |
| 3 | Identity hubs (wallets): participant contexts, credentials, key pairs, DIDs, credential requests |
| 4 | Issuer (authority): holders, attestations, credential definitions, issuance processes, issued credentials |

The same endpoints are in the Postman collection (`../postman/`).

## How reading works in EDC

Every resource has two read endpoints:

| Endpoint | Returns |
|---|---|
| `GET /v3/<resource>/<id>` | one object |
| `POST /v3/<resource>/request` with a **QuerySpec** | a list, filtered, sorted and paginated |

A QuerySpec looks like this:

```json
{"@context": {"@vocab": "https://w3id.org/edc/v0.0.1/ns/"}, "@type": "QuerySpec",
 "filterExpression": [{"operandLeft": "assetId", "operator": "like", "operandRight": "ging-%"}],
 "sortField": "createdAt", "sortOrder": "DESC", "limit": 50, "offset": 0}
```

- `filterExpression` is a list of criteria, **AND-ed**. Operators: `=`, `!=`, `like` (`%` wildcard), `in` (list), `<`, `>`, `<=`, `>=`.
- `operandLeft` is a field path. Plain fields go by name (`id`, `assetId`, `state`, `type`, `correlationId`).
  Asset properties use their full IRI (`properties.'https://w3id.org/edc/v0.0.1/ns/description'`), and nested objects use dots (`contractAgreement.assetId`).
- `state` accepts the name (`FINALIZED`) or the numeric code (`1200`).
- **The default `limit` is small**, so set it explicitly when you want everything.

These operandLeft values were tested on this deployment:

| Resource | Working filter fields |
|---|---|
| assets | `id`, `properties.'<iri>'`, `dataAddress.'<iri>'` |
| policydefinitions, contractdefinitions | `id` |
| contractnegotiations | `state`, `type`, `contractAgreement.assetId`, `correlationId` |
| contractagreements | `assetId` |
| transferprocesses | `state`, `type`, `assetId`, `correlationId`, `contractId` |
| edrs | `assetId`, `agreementId`, `transferProcessId` |

## Helpers

In [1]:
import dataspace as ds

EDC = ds.EDC


def criterion(field, operator, value):
    """One filter condition for a QuerySpec."""
    return {"operandLeft": field, "operator": operator, "operandRight": value}


def query(participant, resource, *conditions, limit=50, offset=0, sort_by=None, descending=True):
    """POST /v3/<resource>/request on a participant's Management API and return the list."""
    query_spec = {"@context": ds.EDC_CONTEXT, "@type": "QuerySpec", "limit": limit, "offset": offset,
                  "filterExpression": list(conditions)}
    if sort_by:
        query_spec["sortField"] = sort_by
        query_spec["sortOrder"] = "DESC" if descending else "ASC"
    return ds.post(f"{ds.management_api(participant)}/{resource}/request", query_spec, expect={200}).json()


def read(participant, resource, object_id):
    """GET /v3/<resource>/<id> on a participant's Management API."""
    return ds.get(f"{ds.management_api(participant)}/{resource}/{object_id}", expect={200}).json()


def table(items, *fields):
    """Print one line per item with the chosen fields."""
    for item in items:
        print("   " + "  ".join(str(item.get(field, "-"))[:40].ljust(40 if field in ("@id", "assetId") else 14) for field in fields))
    print(f"   ({len(items)} items)")

# 1 · Management API

## Assets (provider)

In [2]:
assets = query("provider", "assets", limit=500, sort_by="createdAt")
table(assets[:10], "@id")

POST   /provider/cp/management/v3/assets/request  ->  200
   ging-20s-42930a                         
   ging-travel-42930a                      
   ging-nobody-42930a                      
   ging-data-42930a                        
   ging-20s-1d0d4e                         
   ging-travel-1d0d4e                      
   ging-nobody-1d0d4e                      
   ging-data-1d0d4e                        
   ging-20s-f45c12                         
   ging-travel-f45c12                      
   (10 items)


Filter by id pattern, then read one asset in full (properties **and** its private data address):

In [3]:
ours = query("provider", "assets", criterion("id", "like", "ging-data-%"), limit=500)
table(ours, "@id")

if ours:
    ds.show(read("provider", "assets", ours[0]["@id"]), max_chars=1200)

POST   /provider/cp/management/v3/assets/request  ->  200
   ging-data-f0c3ae                        
   ging-data-7ff0cb                        
   ging-data-2994ed                        
   ging-data-ec6f09                        
   ging-data-2bd706                        
   ging-data-564922                        
   ging-data-d4385d                        
   ging-data-f45c12                        
   ging-data-1d0d4e                        
   ging-data-42930a                        
   (10 items)
GET    /provider/cp/management/v3/assets/ging-data-f0c3ae  ->  200
{
  "@id": "ging-data-f0c3ae",
  "@type": "Asset",
  "properties": {
    "createdAt": "2026-10-09T13:27:14.461Z",
    "name": "ging-data-f0c3ae",
    "description": "readable by consumer",
    "id": "ging-data-f0c3ae",
    "contenttype": "application/json"
  },
  "dataAddress": {
    "@type": "DataAddress",
    "type": "HttpData",
    "proxyQueryParams": "true",
    "baseUrl": "http://provider-backend:8080/api/provide

Filter on an asset property (full IRI) or on its data address:

In [4]:
by_description = query("provider", "assets", criterion(f"properties.'{EDC}description'", "=", "readable by consumer"), limit=500)
print("description = 'readable by consumer':", len(by_description))

blob_assets = query("provider", "assets", criterion(f"dataAddress.'{EDC}type'", "=", "AzureStorage"))
print("AzureStorage assets:", [a["@id"] for a in blob_assets])

POST   /provider/cp/management/v3/assets/request  ->  200
description = 'readable by consumer': 10
POST   /provider/cp/management/v3/assets/request  ->  200
AzureStorage assets: ['azure-blob-storage']


## Policy definitions and contract definitions (provider)

In [5]:
policies = query("provider", "policydefinitions", limit=5, sort_by="createdAt")
table(policies, "@id")
if policies:
    ds.show(policies[0]["policy"], max_chars=800)

POST   /provider/cp/management/v3/policydefinitions/request  ->  200
   8038dbdc-321b-4438-a28b-e0779b78bb3d    
   262cf73a-c182-40f9-a66b-ec38d0ca6a13    
   3c78d081-99c1-43c1-a6bb-01c2265419f5    
   fb5465ab-b630-4f72-9461-f961958a7471    
   b3a82d66-fead-4205-b873-d91003629586    
   (5 items)
{
  "@id": "b7d11d02-d627-4214-ab53-3e629b4313f8",
  "@type": "odrl:Set",
  "odrl:permission": [
    {
      "odrl:action": {
        "@id": "odrl:use"
      },
      "odrl:constraint": {
        "odrl:leftOperand": {
          "@id": "dse-policy:Membership"
        },
        "odrl:operator": {
          "@id": "odrl:eq"
        },
        "odrl:rightOperand": "active"
      }
    },
    {
      "odrl:action": {
        "@id": "odrl:use"
      },
      "odrl:constraint": {
        "odrl:leftOperand": {
          "@id": "edc:inForceDate"
        },
        "odrl:operator": {
          "@id": "odrl:lteq"
        },
        "odrl:rightOperand": "contractAgreement+20s"
      }
    }
  ],
  "o

In [6]:
definitions = query("provider", "contractdefinitions", criterion("id", "like", "contract-%"), limit=5)
for definition in definitions:
    print(f"   {definition['@id']:35} access={definition['accessPolicyId'][:8]}…  selector={definition['assetsSelector']}")

POST   /provider/cp/management/v3/contractdefinitions/request  ->  200
   contract-ging-data-564922           access=5c2cdbcd…  selector={'@type': 'Criterion', 'operandLeft': 'https://w3id.org/edc/v0.0.1/ns/id', 'operator': '=', 'operandRight': 'ging-data-564922'}
   contract-ging-travel-564922         access=813ef407…  selector={'@type': 'Criterion', 'operandLeft': 'https://w3id.org/edc/v0.0.1/ns/id', 'operator': '=', 'operandRight': 'ging-travel-564922'}
   contract-ging-20s-564922            access=b5e815b3…  selector={'@type': 'Criterion', 'operandLeft': 'https://w3id.org/edc/v0.0.1/ns/id', 'operator': '=', 'operandRight': 'ging-20s-564922'}
   contract-ging-data-d4385d           access=96db1fc6…  selector={'@type': 'Criterion', 'operandLeft': 'https://w3id.org/edc/v0.0.1/ns/id', 'operator': '=', 'operandRight': 'ging-data-d4385d'}
   contract-ging-data-f45c12           access=459f8c0a…  selector={'@type': 'Criterion', 'operandLeft': 'https://w3id.org/edc/v0.0.1/ns/id', 'operator':

## Contract negotiations (consumer)

Filter by `state` (name or code), by `type` (`CONSUMER` / `PROVIDER`), or by the asset of the resulting agreement.

In [7]:
finalized = query("consumer", "contractnegotiations", criterion("state", "=", "FINALIZED"), limit=500)
terminated = query("consumer", "contractnegotiations", criterion("state", "=", "TERMINATED"), limit=500)
print(f"FINALIZED: {len(finalized)}   TERMINATED: {len(terminated)}")

table(terminated[:5], "@id", "assetId", "state")

POST   /consumer/cp/management/v3/contractnegotiations/request  ->  200
POST   /consumer/cp/management/v3/contractnegotiations/request  ->  200
FINALIZED: 27   TERMINATED: 3
   1f2a69db-63e7-45fa-82eb-ab16dd7ee777      rest-api-travel-domain-restricted         TERMINATED    
   86132731-bb7a-4e71-b41b-f07517c5ba42      restricted-api                            TERMINATED    
   43a730ca-24a7-4162-8745-0410105367cb      rest-api-travel-domain-restricted         TERMINATED    
   (3 items)


In [8]:
for_our_assets = query("consumer", "contractnegotiations", criterion("contractAgreement.assetId", "like", "ging-%"), limit=500)
table(for_our_assets[:5], "@id", "assetId", "state")

if for_our_assets:
    negotiation_id = for_our_assets[0]["@id"]
    print("\nstate:    ", ds.get(f"{ds.management_api('consumer')}/contractnegotiations/{negotiation_id}/state").json())
    print("agreement:", ds.get(f"{ds.management_api('consumer')}/contractnegotiations/{negotiation_id}/agreement").json().get("@id"))

POST   /consumer/cp/management/v3/contractnegotiations/request  ->  200
   2381f8b8-428b-4a3c-adf1-638f5891902d      ging-data-2994ed                          FINALIZED     
   e8ebf1ae-abd5-4415-8b30-022445c2a21c      ging-20s-2994ed                           FINALIZED     
   2960be60-fc89-4061-8d52-c1fe5d831b87      ging-20s-ec6f09                           FINALIZED     
   0be96aed-fa1c-4cfc-bea3-66be9fa276b9      ging-data-ec6f09                          FINALIZED     
   aab1063f-7556-4d14-8eaf-65af36dc4396      ging-data-564922                          FINALIZED     
   (5 items)
GET    /consumer/cp/management/v3/contractnegotiations/2381f8b8-428b-4a3c-adf1-638f5891902d/state  ->  200

state:     {'@type': 'NegotiationState', 'state': 'FINALIZED', '@context': {'dse-policy': 'https://w3id.org/dse/policy/', '@vocab': 'https://w3id.org/edc/v0.0.1/ns/', 'edc': 'https://w3id.org/edc/v0.0.1/ns/', 'odrl': 'http://www.w3.org/ns/odrl/2/'}}
GET    /consumer/cp/management/v3/contractnegot

## Contract agreements

Each side has its own copy, under its own `@id`. `GET /v3/contractagreements/<id>/negotiation` goes back to the negotiation that produced it.

In [9]:
consumer_agreements = query("consumer", "contractagreements", criterion("assetId", "like", "ging-%"), limit=500)
provider_agreements = query("provider", "contractagreements", criterion("assetId", "like", "ging-%"), limit=500)
print(f"consumer copies: {len(consumer_agreements)}   provider copies: {len(provider_agreements)}")
table(consumer_agreements[:5], "@id", "assetId", "agreementId")

POST   /consumer/cp/management/v3/contractagreements/request  ->  200


POST   /provider/cp/management/v3/contractagreements/request  ->  200
consumer copies: 14   provider copies: 14
   ad09c5ad-3b7e-4cc6-afaf-35d6d7f20574      ging-data-2994ed                          1f462ba0-0165-4bfb-b710-2b1dc6fdc8f8
   37cc09b1-b01d-4567-b0c8-ee73357fc423      ging-20s-2994ed                           96565243-9c28-4d4e-b968-4320dc526ea6
   dcbcdf4d-bf68-44ba-a91a-72f03b08b39a      ging-20s-ec6f09                           2a2d57dc-95a3-4317-a523-250c15785ce8
   4b2db84a-923c-4140-8ed0-01f069cfb729      ging-data-ec6f09                          4f0a2d29-7d03-4bf7-a615-e4a4ce171f6a
   7b4fe8c2-4ca9-4191-8848-aa04e4dd359d      ging-data-564922                          890246bb-fb70-45fc-a03e-22a6384fcf4d
   (5 items)


In [10]:
if consumer_agreements:
    agreement_id = consumer_agreements[0]["@id"]
    negotiation = ds.get(f"{ds.management_api('consumer')}/contractagreements/{agreement_id}/negotiation").json()
    print("produced by negotiation", negotiation["@id"], "->", negotiation["state"])

GET    /consumer/cp/management/v3/contractagreements/ad09c5ad-3b7e-4cc6-afaf-35d6d7f20574/negotiation  ->  200
produced by negotiation 2381f8b8-428b-4a3c-adf1-638f5891902d -> FINALIZED


## Transfer processes

Both sides keep one record per transfer. On the provider, `correlationId` is the consumer's transfer id; on the consumer, the provider's.

In [11]:
consumer_transfers = query("consumer", "transferprocesses", limit=500, sort_by="createdAt")
by_state = {}
for transfer in consumer_transfers:
    by_state[transfer["state"]] = by_state.get(transfer["state"], 0) + 1
print("consumer transfers by state:", by_state)

table(consumer_transfers[:5], "@id", "assetId", "state", "transferType")

POST   /consumer/cp/management/v3/transferprocesses/request  ->  200
consumer transfers by state: {'STARTED': 18, 'TERMINATED': 18, 'COMPLETED': 1}
   f9637f24-0a85-4d4d-803c-3685402f115e      ging-data-564922                          STARTED         HttpData-PULL 
   38c6d640-8758-494b-a814-c85338a31432      ging-data-564922                          TERMINATED      HttpData-PULL 
   61bcbd2c-457a-4166-bc80-b51c4ba52950      ging-20s-1d0d4e                           TERMINATED      HttpData-PULL 
   73356fdd-7f15-42ec-a799-84f33551a93e      ging-data-1d0d4e                          STARTED         HttpData-PULL 
   f5a76299-6c79-4bcf-b248-e840f3412a99      ging-data-1d0d4e                          TERMINATED      HttpData-PULL 
   (5 items)


In [12]:
running = query("consumer", "transferprocesses", criterion("state", "=", "STARTED"), limit=500)
print(len(running), "transfers STARTED on the consumer")

if running:
    provider_side = query("provider", "transferprocesses", criterion("correlationId", "=", running[0]["@id"]))
    table(provider_side, "@id", "assetId", "state", "type")

POST   /consumer/cp/management/v3/transferprocesses/request  ->  200
18 transfers STARTED on the consumer


POST   /provider/cp/management/v3/transferprocesses/request  ->  200
   c04b170c-962e-48b5-9dfc-163f4a714a57      rest-api                                  STARTED         PROVIDER      
   (1 items)


## EDRs (consumer)

One entry per pull transfer whose EDR is still held. `GET /v3/edrs/<transferProcessId>/dataaddress` returns the endpoint and token.

In [13]:
edrs = query("consumer", "edrs", criterion("assetId", "like", "ging-%"), limit=500)
table(edrs[:5], "transferProcessId", "assetId", "agreementId")

if edrs:
    edr = ds.get(f"{ds.management_api('consumer')}/edrs/{edrs[0]['transferProcessId']}/dataaddress").json()
    print("\nendpoint:", edr["endpoint"], "| token claims:", ds.decode_jwt_payload(edr["authorization"]))

POST   /consumer/cp/management/v3/edrs/request  ->  200
   0dcd5c82-1e39-45d7-955e-c7890418e312  ging-data-2994ed                          ad09c5ad-3b7e-4cc6-afaf-35d6d7f20574
   035b7fce-f1c7-4288-bbc3-085f4e5fcfeb  ging-data-ec6f09                          4b2db84a-923c-4140-8ed0-01f069cfb729
   6948bc0f-5e54-43bb-a5e9-ed16713a46cc  ging-data-f0c3ae                          dc45aa4c-2d09-4088-9da9-15de85d0b013
   30d1ffd5-baac-43bc-bbe9-258a9a913e79  ging-data-2bd706                          0c55ef18-531a-4309-ae00-ac1f19c2e7c2
   9e89da79-0a50-41a2-add9-f9d22a08b3bb  ging-data-564922                          7b4fe8c2-4ca9-4191-8848-aa04e4dd359d
   (5 items)


GET    /consumer/cp/management/v3/edrs/0dcd5c82-1e39-45d7-955e-c7890418e312/dataaddress  ->  200

endpoint: http://provider-dataplane:8181/api/public/ | token claims: {'iss': 'did:web:provider-identityhub%3A8383:api:did', 'aud': 'did:web:consumer-identityhub%3A8383:api:did', 'sub': 'did:web:provider-identityhub%3A8383:api:did', 'iat': 1791552074, 'jti': '37e9956d-bb21-49f4-8898-b316b59a80eb'}


With that token you can call the **provider's** public data plane directly, which is what the consumer gateway does for you.
The token goes in `Authorization` as is, with no `Bearer` prefix.

In [14]:
if edrs:
    direct = ds.get(f"{ds.BASE_URL}/provider/dp/public/", params={"message": "direct call"},
                    headers={"Authorization": edr["authorization"]})
    print(direct.text)

GET    /provider/dp/public/  ->  200
{"message":"direct call"}


## Data planes, retirements and secrets

In [15]:
for data_plane in ds.get(f"{ds.management_api('provider')}/dataplanes").json():
    print("   ", data_plane["url"], data_plane["state"])
    print("    transfer types:", data_plane["allowedTransferTypes"])

GET    /provider/cp/management/v3/dataplanes  ->  200
    http://provider-dataplane:8383/api/control/v1/dataflows REGISTERED
    transfer types: ['AzureStorage-PUSH', 'HttpData-PULL-HttpData', 'HttpData-PUSH-HttpData', 'HttpData-PULL', 'HttpData-PUSH', 'AzureStorage-PUSH-HttpData']


In [16]:
retirements = query("provider", "contractagreements/retirements", limit=50)
print(len(retirements), "retired agreements")
for retirement in retirements[:3]:
    print("   ", {key.rsplit("/", 1)[1]: value for key, value in retirement.items()})

POST   /provider/cp/management/v3/contractagreements/retirements/request  ->  200
0 retired agreements


`GET /v3/secrets/<id>` returns the secret **in clear text**: anyone with access to the Management API can read the participant's secrets.

In [17]:
ds.show(ds.get(f"{ds.management_api('provider')}/secrets/oauth-secret", expect={200, 404}).json(), max_chars=300)

GET    /provider/cp/management/v3/secrets/oauth-secret  ->  200
{
  "@id": "oauth-secret",
  "@type": "Secret",
  "value": "supersecret",
  "@context": {
    "dse-policy": "https://w3id.org/dse/policy/",
    "@vocab": "https://w3id.org/edc/v0.0.1/ns/",
    "edc": "https://w3id.org/edc/v0.0.1/ns/",
    "odrl": "http://www.w3.org/ns/odrl/2/"
  }
}


# 2 · Catalogs

## DSP catalog and a single dataset (as the consumer)

`/v3/catalog/request` returns the whole catalog. `/v3/catalog/dataset/request` asks the provider for one dataset by id.

In [18]:
def provider_catalog(query_spec=None):
    catalog_request = {"@context": ds.EDC_CONTEXT, "@type": "CatalogRequest",
                       "counterPartyAddress": ds.dsp_address("provider"), "counterPartyId": ds.did("provider"),
                       "protocol": ds.DSP_PROTOCOL, "querySpec": query_spec or {"limit": 500}}
    return ds.post(f"{ds.management_api('consumer')}/catalog/request", catalog_request, expect={200}).json()


catalog = provider_catalog()
print(len(ds.catalog_datasets(catalog)), "datasets visible to the consumer")

POST   /consumer/cp/management/v3/catalog/request  ->  200
52 datasets visible to the consumer


In [19]:
some_dataset_id = ds.catalog_datasets(catalog)[0]["@id"]
dataset_request = {"@context": ds.EDC_CONTEXT, "@type": "DatasetRequest", "@id": some_dataset_id,
                   "counterPartyAddress": ds.dsp_address("provider"), "counterPartyId": ds.did("provider"),
                   "protocol": ds.DSP_PROTOCOL}
dataset = ds.post(f"{ds.management_api('consumer')}/catalog/dataset/request", dataset_request, expect={200}).json()
print(dataset["@id"], "| offers:", len(ds.as_list(dataset["hasPolicy"])), "| formats:", [d["format"] for d in ds.as_list(dataset["distribution"])])

POST   /consumer/cp/management/v3/catalog/dataset/request  ->  200
rest-api | offers: 1 | formats: ['AzureStorage-PUSH', 'HttpData-PULL', 'HttpData-PUSH']


## Federated catalog: direct (authority cache) and filtered (as the consumer)

In [20]:
federated = ds.post(f"{ds.BASE_URL}/authority/catalog/v1alpha/catalog/query",
                    {"@context": ds.EDC_CONTEXT, "@type": "QuerySpec", "limit": 500}, expect={200}).json()
filtered = ds.post(f"{ds.BASE_URL}/consumer/cp/management/v1alpha/catalog/query", {}, expect={200}).json()

for label, catalogs in [("direct", federated), ("filtered for consumer", filtered)]:
    print(f"{label}:")
    for catalog_entry in catalogs:
        print(f"   {ds.catalog_owner(catalog_entry)}: {len(ds.catalog_datasets(catalog_entry))} datasets")

POST   /authority/catalog/v1alpha/catalog/query  ->  200


POST   /consumer/cp/management/v1alpha/catalog/query  ->  200
direct:
   did:web:provider-identityhub%3A8383:api:did: 52 datasets
   did:web:consumer-identityhub%3A8383:api:did: 0 datasets
filtered for consumer:
   did:web:provider-identityhub%3A8383:api:did: 50 datasets
   did:web:consumer-identityhub%3A8383:api:did: 0 datasets


# 3 · Identity hubs (wallets)

Two kinds of endpoint:

- per participant context: `/<p>/ih/identity/v1alpha/participants/<context>/...`;
- across every context of that identity hub (admin): `/<p>/ih/identity/v1alpha/{participants|credentials|keypairs|dids}`.

In [21]:
for participant in ds.PARTICIPANTS:
    contexts = ds.get(f"{ds.BASE_URL}/{participant}/ih/identity/v1alpha/participants", quiet=True).json()
    for context in contexts:
        state = {100: "CREATED", 200: "ACTIVATED", 300: "DEACTIVATED"}.get(context["state"], context["state"])
        print(f"   {participant:10} context {context['participantContextId']}  state={state}  roles={context['roles']}")

   authority  context did:web:authority-identityhub%3A8383:api:did  state=ACTIVATED  roles=['admin']
   provider   context did:web:provider-identityhub%3A8383:api:did  state=ACTIVATED  roles=['admin']
   consumer   context did:web:consumer-identityhub%3A8383:api:did  state=ACTIVATED  roles=['admin']


In [22]:
CREDENTIAL_STATES = {500: "ISSUED", 600: "REVOKED", 700: "SUSPENDED", 800: "EXPIRED"}

for participant in ds.PARTICIPANTS:
    stored = ds.get(f"{ds.identity_api(participant)}/credentials", quiet=True).json()
    summary = [f"{c['verifiableCredential']['credential']['type'][-1]}({CREDENTIAL_STATES.get(c['state'], c['state'])})" for c in stored]
    print(f"   {participant:10} {len(stored)} credentials: {summary}")

   authority  11 credentials: ['MembershipCredential(ISSUED)', 'MembershipCredential(ISSUED)', 'MembershipCredential(ISSUED)', 'MembershipCredential(ISSUED)', 'DomainCredential(ISSUED)', 'DomainCredential(ISSUED)', 'DomainCredential(ISSUED)', 'DomainCredential(ISSUED)', 'MembershipCredential(ISSUED)', 'VisibilityCredential(ISSUED)', 'BitstringStatusListCredential(ISSUED)']
   provider   2 credentials: ['MembershipCredential(ISSUED)', 'DomainCredential(ISSUED)']
   consumer   4 credentials: ['MembershipCredential(ISSUED)', 'DomainCredential(ISSUED)', 'VisibilityCredential(ISSUED)', 'MembershipCredential(ISSUED)']


In [23]:
KEY_PAIR_STATES = {100: "CREATED", 200: "ACTIVATED", 300: "ROTATED", 400: "REVOKED"}

for key_pair in ds.get(f"{ds.identity_api('provider')}/keypairs").json():
    print("   key", key_pair["keyId"])
    print("      private key alias in Vault:", key_pair["privateKeyAlias"], "| state:", KEY_PAIR_STATES.get(key_pair["state"], key_pair["state"]))

# dids/query returns the DID documents this identity hub publishes
for did_document in ds.post(f"{ds.identity_api('provider')}/dids/query", {}).json():
    print("   DID", did_document["id"], "| services:", [s["type"] for s in did_document.get("service", [])])

GET    /provider/ih/identity/v1alpha/participants/ZGlkOndlYjpwcm92aWRlci1pZGVudGl0eWh1YiUzQTgzODM6YXBpOmRpZA==/keypairs  ->  200
   key did:web:provider-identityhub%3A8383:api:did#my-key
      private key alias in Vault: provider | state: ACTIVATED
POST   /provider/ih/identity/v1alpha/participants/ZGlkOndlYjpwcm92aWRlci1pZGVudGl0eWh1YiUzQTgzODM6YXBpOmRpZA==/dids/query  ->  200
   DID did:web:provider-identityhub%3A8383:api:did | services: ['CredentialService', 'DSPMessaging']


# 4 · Issuer (authority)

All under the authority's issuer context. List endpoints are `POST .../<resource>/query` with `{}`.

Holders cannot be read by DID in the path: the `%3A` in the DID makes the server reject the URL. Use `holders/query` instead.

In [24]:
issuer = ds.issuer_admin_api()

holders = ds.post(f"{issuer}/holders/query", {}).json()
print("holders:", [(h["holderName"], h["did"]) for h in holders])

attestations = ds.post(f"{issuer}/attestations/query", {}).json()
print("attestation definitions:", [(a["id"], a["attestationType"]) for a in attestations])

definitions = ds.post(f"{issuer}/credentialdefinitions/query", {}).json()
print("credential definitions:", [(d["id"], d["credentialType"]) for d in definitions])

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/holders/query  ->  200
holders: [('provider', 'did:web:provider-identityhub%3A8383:api:did'), ('consumer', 'did:web:consumer-identityhub%3A8383:api:did'), ('authority', 'did:web:authority-identityhub%3A8383:api:did')]


POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestations/query  ->  200
attestation definitions: [('membership-attestation-def-1', 'json-database'), ('domain-attestation-def-1', 'database'), ('visibility-attestation-def-1', 'json-database')]
POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/credentialdefinitions/query  ->  200
credential definitions: [('membership-credential-def-1', 'MembershipCredential'), ('domain-credential-def-1', 'DomainCredential'), ('visibility-credential-def-1', 'VisibilityCredential')]


In [25]:
from collections import Counter

processes = ds.post(f"{issuer}/issuanceprocesses/query", {}).json()
print("issuance processes:", Counter(p["state"] for p in processes))

issued = ds.post(f"{issuer}/credentials/query", {}).json()
print("credentials issued:", len(issued))
for entry in issued[:3]:
    credential = entry["credential"]
    print("   ", credential["type"][-1], "->", ds.as_list(credential["credentialSubject"])[0]["id"])

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/issuanceprocesses/query  ->  200
issuance processes: Counter({'DELIVERED': 8, 'ERRORED': 7})


POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/credentials/query  ->  200
credentials issued: 11
    MembershipCredential -> did:web:consumer-identityhub%3A8383:api:did
    MembershipCredential -> did:web:authority-identityhub%3A8383:api:did
    MembershipCredential -> did:web:authority-identityhub%3A8383:api:did


In [26]:
for kind in ["attestation-membership", "attestation-domain", "attestation-visibility"]:
    rows = ds.post(f"{issuer}/{kind}/request", {}).json()
    print(f"{kind}: {len(rows)} rows")
    for row in rows[:3]:
        print("   ", row)

POST   /authority/is/issueradmin/v1alpha/participants/ZGlkOndlYjphdXRob3JpdHktaWRlbnRpdHlodWIlM0E4MzgzOmFwaTpkaWQ=/attestation-membership/request  ->  200
attestation-membership: 3 rows
    {'id': 'did:web:provider-identityhub%3A8383:api:did', 'holderId': 'did:web:provider-identityhub%3A8383:api:did', 'name': 'provider', 'membershipType': 'FullMember', 'membershipStartDate': '2026-10-09T12:38:14.343Z', 'properties': {'companySegment': 'Airlines'}}
    {'id': 'did:web:consumer-identityhub%3A8383:api:did', 'holderId': 'did:web:consumer-identityhub%3A8383:api:did', 'name': 'consumer', 'membershipType': 'FullMember', 'membershipStartDate': '2026-10-09T12:38:14.496Z', 'properties': {'companySegment': 'Airlines'}}
    {'id': 'did:web:authority-identityhub%3A8383:api:did', 'holderId': 'did:web:authority-identityhub%3A8383:api:did', 'name': 'authority', 'membershipType': 'FullMember', 'membershipStartDate': '2026-10-09T12:38:14.561Z', 'properties': {'companySegment': 'Airlines'}}
POST   /autho

# Your turn

Change the conditions and run again. Some ideas:

- every terminated transfer of the consumer, newest first: `query("consumer", "transferprocesses", criterion("state", "=", "TERMINATED"), sort_by="stateTimestamp")`
- the provider's view of the negotiations: `query("provider", "contractnegotiations", criterion("type", "=", "PROVIDER"))`
- assets created in your run: `query("provider", "assets", criterion("id", "like", "%" + ds.recall("run_id", "00_environment.ipynb")))`

In [27]:
results = query("consumer", "transferprocesses", criterion("state", "=", "TERMINATED"), limit=10, sort_by="stateTimestamp")
table(results, "@id", "assetId", "state")

POST   /consumer/cp/management/v3/transferprocesses/request  ->  200
   38c6d640-8758-494b-a814-c85338a31432      ging-data-564922                          TERMINATED    
   61bcbd2c-457a-4166-bc80-b51c4ba52950      ging-20s-1d0d4e                           TERMINATED    
   f5a76299-6c79-4bcf-b248-e840f3412a99      ging-data-1d0d4e                          TERMINATED    
   5f1ed6d7-7017-42dd-8646-ce11a1344e7a      ging-20s-f45c12                           TERMINATED    
   064fcbd4-74bf-4b35-82da-b7c7daf379e6      ging-data-f45c12                          TERMINATED    
   9f245845-f470-4fbf-ba3f-e9391be78063      ging-20s-564922                           TERMINATED    
   65bb3209-a18c-4a84-a721-d1e0c3357023      ging-data-564922                          TERMINATED    
   53a64014-5830-4ffc-a244-47c8631d520a      ging-20s-2bd706                           TERMINATED    
   fd41603d-d5f2-40d3-b212-bed410e07b26      ging-data-2bd706                          TERMINATED    
   f7e3fd73-3